In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf


df = yf.download("GC=F", start="2026-01-01", end="2026-05-30")
print(df.head())

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='gold', linewidth=2)
plt.title("Gold Price - 2026")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
war_events = {
    "2026-02-28": "War Starts",
    "2026-03-09": "Hormuz Threat",
    "2026-04-08": "Ceasefire"
}

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='gold', linewidth=2)

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date),
                color='red', linestyle='--', alpha=0.8)
    plt.text(pd.to_datetime(date),
             df['Close'].max() * 0.98,
             label, rotation=45, fontsize=9, color='red')

plt.title("Gold Price During 2026 Iran War")
plt.grid(True, alpha=0.3)
plt.show()

##### Finding 1 — Gold Behaviour During War

**Common belief:** Gold always rises during war (safe haven asset)

**What actually happened in 2026:**
Gold peaked at $5,595 on January 29, 2026 — an all-time high —
*before* the war even started on February 28.

When US-Israel strikes on Iran began, gold actually dropped.

**Why?**
Investors anticipated the conflict weeks in advance and bought gold early.
By the time war started, the price was already "priced in."
Smart institutional investors sold their positions to lock in profits.

**This is called: "Buy the Rumor, Sell the News"**

> This finding challenges the common assumption that war = gold price rise.
> The real signal was in the weeks BEFORE the conflict, not after.



In [ ]:
pre_war = df.loc['2026-02-01':'2026-02-27', ('Close', 'GC=F')].mean()
peak = df.loc['2026-03-01':'2026-03-10', ('Close', 'GC=F')].max()
spike = ((peak - pre_war) / pre_war) * 100

print(f"Pre-war avg gold price: ${pre_war:.2f}")
print(f"Peak gold price: ${peak:.2f}")
print(f"Gold spiked: {spike:.1f}% in first 10 days of war")

In [ ]:
# Download oil alongside gold
oil = yf.download("CL=F", start="2026-01-01", end="2026-05-30")
sp500 = yf.download("^GSPC", start="2026-01-01", end="2026-05-30")
gld_etf = yf.download("GLD", start="2026-01-01", end="2026-05-30")  # Gold ETF — this shows investor flows
vix = yf.download("^VIX", start="2026-01-01", end="2026-05-30")     # Fear index

print("Oil data:", oil.shape)
print("SP500 data:", sp500.shape)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(oil.index, oil['Close'], color='black', linewidth=2)

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date), color='red', linestyle='--', alpha=0.8)
    plt.text(pd.to_datetime(date), oil['Close'].max()*0.98,
             label, rotation=45, fontsize=9, color='red')

plt.title("WTI Oil Price During 2026 Iran War")
plt.grid(True, alpha=0.3)
plt.show()

## Finding 2 — Oil Was The Real War Trade, Not Gold

**What the data shows (Feb 28 → Mar 10, 2026):**
- Oil (WTI): **+24.5%**
- Gold: **-0.0%** (flat)
- S&P 500: **-1.4%**

**Key Insight:**
Contrary to popular belief, gold was NOT the biggest
mover when war started. Oil was.

**Why Oil and not Gold?**
This war had a unique characteristic — Iran physically
blocked the Strait of Hormuz, which carries 20% of
world oil supply. Investors immediately priced in an
oil supply shock, not just geopolitical fear.

Gold was already flat because smart money had bought
it weeks before (Finding 1 — Buy the Rumor).

**The real investor trade was: Short S&P 500 + Buy Oil**

> "In most wars, gold wins. In this war, oil won
>  because the battlefield was an oil chokepoint."

In [ ]:
# Combining everything into one dataframe
import pandas as pd

flows = pd.DataFrame({
    'Gold_Price': df[('Close', 'GC=F')],
    'Oil_Price': oil[('Close', 'CL=F')],
    'SP500_Price': sp500[('Close', '^GSPC')],
    'Gold_ETF_Volume': gld_etf[('Volume', 'GLD')],  # investors rushing INTO gold
    'SP500_Volume': sp500[('Volume', '^GSPC')],         # investors rushing OUT of stocks
    'Fear_VIX': vix[('Close', '^VIX')],
})


# This shows % change from starting point
base = flows['2026-01-01':'2026-02-27']
flows_normalized = (flows / flows.iloc[0]) * 100

# Plotting all assets together
plt.figure(figsize=(16, 7))
plt.plot(flows_normalized['Gold_Price'], label='Gold', color='gold', linewidth=2)
plt.plot(flows_normalized['Oil_Price'], label='Oil', color='black', linewidth=2)
plt.plot(flows_normalized['SP500_Price'], label='S&P 500', color='blue', linewidth=2)
plt.plot(flows_normalized['Fear_VIX'], label='Fear Index (VIX)',
         color='red', linewidth=2, linestyle='--')

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date), color='red', linestyle=':', alpha=0.6)

plt.axhline(100, color='gray', linestyle='-', alpha=0.3)  # baseline
plt.title("Capital Flows — Where Did Investor Money Go During 2026 Iran War?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# How much did SP500 lose vs how much gold gained
pre_war_date = '2026-02-27'
peak_war_date = '2026-03-10'

sp500_change = ((sp500[('Close', '^GSPC')][peak_war_date] - sp500[('Close', '^GSPC')][pre_war_date])
                / sp500[('Close', '^GSPC')][pre_war_date] * 100)

gold_change = ((df[('Close', 'GC=F')][peak_war_date] - df[('Close', 'GC=F')][pre_war_date])
               / df[('Close', 'GC=F')][pre_war_date] * 100)

oil_change = ((oil[('Close', 'CL=F')][peak_war_date] - oil[('Close', 'CL=F')][pre_war_date])
              / oil[('Close', 'CL=F')][pre_war_date] * 100)

print(f"S&P 500 changed: {sp500_change:.1f}%")
print(f"Gold changed: {gold_change:.1f}%")
print(f"Oil changed: {oil_change:.1f}%")

In [ ]:
import seaborn as sns

corr = flows[['Gold_Price', 'Oil_Price',
              'SP500_Price', 'Fear_VIX']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm',
            center=0, fmt='.2f')
plt.title("Asset Correlation During 2026 Iran War")
plt.show()

## Finding 3 — Correlation Analysis: What Really Moved With Fear?

**SP500 vs Fear (VIX): -0.71** — Strongest relationship
Every spike in fear directly caused a stock market selloff.
Investors panic-sold equities throughout the war period.

**Oil vs Fear (VIX): +0.41** — Moderate relationship  
Fear reliably pushed oil prices higher.
The Strait of Hormuz threat made oil the primary fear trade.

**Gold vs Oil: -0.33** — Negative relationship
Surprising finding — gold and oil moved in OPPOSITE directions.
Investors rotated between the two rather than buying both.
Oil won the rotation due to direct supply chain threat.

**Gold vs Fear (VIX): +0.16** — Almost zero relationship
Gold showed almost no response to fear during the war.
This confirms Finding 1 — gold was already "spent"
having peaked at $5,595 a month before war started.

### Summary: The War Trade Hierarchy
1. SELL stocks (fear drove -0.71 correlation)
2. BUY oil (Hormuz supply shock, +0.41 with fear)  
3. Gold irrelevant during active war phase

In [ ]:
pre_war = flows['2026-01-01':'2026-02-27']
during_war = flows['2026-02-28':'2026-04-08']
ceasefire = flows['2026-04-09':'2026-05-30']

vol_table = pd.DataFrame({
    'Pre War': pre_war[['Gold_Price','Oil_Price',
                        'SP500_Price','Fear_VIX']].std(),
    'During War': during_war[['Gold_Price','Oil_Price',
                              'SP500_Price','Fear_VIX']].std(),
    'Ceasefire': ceasefire[['Gold_Price','Oil_Price',
                            'SP500_Price','Fear_VIX']].std()
})

print(vol_table)

## Finding 4 — Volatility Analysis: War vs Ceasefire

| Asset | Pre-War | During War | Ceasefire |
|-------|---------|------------|-----------|
| Gold | 271.6 | 289.4 | 111.3 |
| Oil | 2.98 | 10.60 | 6.09 |
| S&P 500 | 50.4 | 141.5 | 210.3 |
| Fear VIX | 2.10 | 2.56 | 1.03 |

**Oil** — Most dramatic war impact. Volatility tripled
instantly when Strait of Hormuz was blocked.
Even ceasefire couldn't fully restore stability.

**Fear (VIX)** — Spiked during war, then collapsed
BELOW pre-war levels during ceasefire.
Markets became calmer than before the war started.

**Gold** — Barely responded to war at all.
Ceasefire caused gold volatility to collapse by 60%.
Investors had zero interest in gold once fear left.

### 🚨 Most Surprising Finding
**S&P 500 was most volatile DURING ceasefire, not war.**

During war: investors just sold and waited.
During ceasefire: everyone rushed back in simultaneously
causing massive daily swings — known as a
"Relief Rally Volatility Spike."

> War makes investors freeze.
> Peace makes them stampede.

In [ ]:
lmt = yf.download("LMT", start="2026-01-01", end="2026-05-30")
rtx = yf.download("RTX", start="2026-01-01", end="2026-05-30")

In [ ]:
# Normalizing everything to 100 for fair comparison
defense = pd.DataFrame({
    'Lockheed_Martin': (lmt[('Close', 'LMT')] / lmt[('Close', 'LMT')].iloc[0]) * 100,
    'Raytheon': (rtx[('Close', 'RTX')] / rtx[('Close', 'RTX')].iloc[0]) * 100,
    'SP500': (sp500[('Close', '^GSPC')] / sp500[('Close', '^GSPC')].iloc[0]) * 100,
})

# Plotting
plt.figure(figsize=(14,6))
plt.plot(defense['Lockheed_Martin'],
         label='Lockheed Martin (LMT)', color='green', linewidth=2)
plt.plot(defense['Raytheon'],
         label='Raytheon (RTX)', color='blue', linewidth=2)
plt.plot(defense['SP500'],
         label='S&P 500', color='red', linewidth=2, linestyle='--')

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date),
                color='red', linestyle=':', alpha=0.6)
    plt.text(pd.to_datetime(date),
             defense.max().max() * 0.98,
             label, rotation=45, fontsize=8, color='red')

plt.axhline(100, color='gray', linestyle='-', alpha=0.3)
plt.title("Defense Stocks vs S&P 500 — Who Made Money During War?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Printing exact numbers
pre = '2026-02-27'
peak = '2026-03-10'

lmt_change = ((lmt[('Close', 'LMT')][peak] - lmt[('Close', 'LMT')][pre]) / lmt[('Close', 'LMT')][pre] * 100)
rtx_change = ((rtx[('Close', 'RTX')][peak] - rtx[('Close', 'RTX')][pre]) / rtx[('Close', 'RTX')][pre] * 100)
sp_change = ((sp500[('Close', '^GSPC')][peak] - sp500[('Close', '^GSPC')][pre]) / sp500[('Close', '^GSPC')][pre] * 100)

print(f"Lockheed Martin: {float(lmt_change):.1f}%")
print(f"Raytheon: {float(rtx_change):.1f}%")
print(f"S&P 500: {float(sp_change):.1f}%")

## Finding 5 — Even Defense Stocks Didn't Win

**Feb 28 → Mar 10, 2026:**
- Raytheon (RTX): +2.2%
- Lockheed Martin (LMT): -0.5%
- S&P 500: -1.4%

**What this means:**
Common belief — war always benefits defense companies.
Reality — Raytheon barely outperformed, LMT actually lost value.

**Why?**
This war moved TOO fast for defense stock investors.
The conflict was air strikes and naval blockades —
not a prolonged ground war requiring massive weapons orders.
Investors couldn't price in long-term defense contracts
when nobody knew if the war would last weeks or months.

**The only clear winner across all 5 findings was Oil.**
Everything else either fell, stayed flat, or gave mixed signals.

> "In the 2026 Iran War, the only trade that worked
>  was the one closest to the battlefield — oil."

# Conclusion — What the 2026 Iran War Taught Us About
# Investor Behavior

## The 5 Key Findings

### Finding 1 — Buy the Rumor, Sell the News
Gold peaked at $5,595 on January 29 — a full month
before war started. By the time bombs dropped,
smart money had already exited their gold positions.
Lesson: Markets price in anticipated events early.
Waiting for confirmation means you already missed the trade.

### Finding 2 — Oil Was The Only Real War Trade
Oil surged +24.5% when war started.
Gold: flat. Stocks: slightly down. Defense: barely moved.
The Strait of Hormuz blockade created a direct,
undeniable supply shock that investors couldn't ignore.
Lesson: Trade the specific economic impact, not the
general "war = buy gold" narrative.

### Finding 3 — Fear Killed Stocks, Not Gold
Correlation of Fear (VIX) with S&P 500: -0.71
Correlation of Fear (VIX) with Gold: +0.16
Fear during this war expressed itself through
stock selloffs, not gold buying.
Lesson: The relationship between fear and assets
changes depending on the TYPE of conflict.

### Finding 4 — Peace Is More Chaotic Than War
S&P 500 volatility: 50 (pre-war) → 141 (war) → 210 (ceasefire)
The relief rally after ceasefire caused MORE stock
market swings than the war itself.
Lesson: Ceasefire announcements create stampede buying —
some of the biggest single-day moves happen when fear leaves,
not when it arrives.

### Finding 5 — Defense Stocks Are A Myth
LMT: -0.5% | RTX: +2.2% | S&P 500: -1.4%
Defense stocks barely outperformed during this war.
Short, sharp air-strike conflicts don't generate
the long-term weapons contracts defense investors need.
Lesson: Defense stock thesis only works in prolonged
ground wars, not precision strike campaigns.

---

## Overall Conclusion

**The 2026 Iran War broke almost every textbook rule:**
- Gold didn't spike when war started ❌
- Defense stocks didn't rally ❌  
- Fear didn't drive gold ❌
- Oil spiked massively ✅
- Markets were calmer after war than before ✅ (unexpected)

**What actually drove markets was not the war itself —
it was the specific geographic chokepoint:
The Strait of Hormuz.**

One strait. 20% of world oil supply.
That single fact explained more about 2026 market
behavior than any traditional war playbook.

---
*Analysis conducted using Python, yfinance, Pandas*
*Data range: January 1, 2026 — May 30, 2026*
*Assets analyzed: Gold, WTI Oil, S&P 500, VIX,*
*Lockheed Martin, Raytheon*